# Kaggle: parallel learning-curve experiments on T4 x2

This notebook runs a configurable experiment matrix on Kaggle's **two physical T4 GPUs**. `EXPERIMENTS_PER_GPU=4` means eight concurrent training processes (four sharing each GPU), not eight GPUs. If CUDA runs out of memory or throughput gets worse, reduce it to 2 or 1.

The dashboard contains one progress bar per process slot, an overall completion bar, GPU occupancy, queued experiments and failures. Learning curves are logged separately to W&B at `LEARNING_CURVE_EVERY_STEPS`.

Before running: select **GPU T4 x2**, enable Internet, and add `WANDB_API_KEY` under **Add-ons > Secrets**.

In [ ]:
# ---- Experiment configuration: edit only this cell ----
K_UNITS = 64
TOTAL_EXPERIMENTS = 24       # first N experiments from the matrix (maximum 24)
EXPERIMENTS_PER_GPU = 4      # 4 x 2 physical GPUs = 8 concurrent processes
MAX_EPOCHS = 100
LEARNING_CURVE_EVERY_STEPS = 250  # W&B train/validation curves
DASHBOARD_EVERY_STEPS = 10        # progress only; does not add W&B points
GPUS = [0, 1]
WANDB_PROJECT = 'pc_dataset_eval'
OUTPUT_ROOT = '/kaggle/working/results'

REPOSITORY_URL = 'https://github.com/GianpieroTulipani/cirkit-circuits-training.git'
REPOSITORY_REF = None  # set a commit SHA for an exactly reproducible run

assert K_UNITS > 0
assert 1 <= TOTAL_EXPERIMENTS <= 24
assert EXPERIMENTS_PER_GPU > 0
assert MAX_EPOCHS > 0
assert LEARNING_CURVE_EVERY_STEPS > 0 and DASHBOARD_EVERY_STEPS > 0

In [ ]:
# Prepare the repository and dependencies. Kaggle Internet must be enabled.
from pathlib import Path
import os
import subprocess
import sys

workspace = Path('/kaggle/working')
repository = workspace / 'cirkit-circuits-training'
if not repository.exists():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repository)], check=True)
elif REPOSITORY_REF is None:
    subprocess.run(['git', 'pull', '--ff-only'], cwd=repository, check=True)
if REPOSITORY_REF:
    subprocess.run(['git', 'fetch', '--all', '--tags'], cwd=repository, check=True)
    subprocess.run(['git', 'checkout', REPOSITORY_REF], cwd=repository, check=True)
required_files = [
    repository / 'scripts' / 'run_learning_curves.py',
    repository / 'scripts' / 'evaluate_image_circuit.py',
    repository / 'notebooks' / 'data-driven-initialization.ipynb',
]
missing_files = [str(path.relative_to(repository)) for path in required_files if not path.exists()]
if missing_files:
    raise RuntimeError(
        'The selected GitHub revision does not contain the parallel launcher: '
        + ', '.join(missing_files)
        + '. Commit and push the local changes, then restart this notebook.'
    )
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=repository, text=True).strip()
print(f'Using repository commit: {commit}')
os.chdir(repository)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)

from kaggle_secrets import UserSecretsClient
import wandb
wandb_key = UserSecretsClient().get_secret('WANDB_API_KEY')
if not wandb_key or not wandb_key.strip():
    raise RuntimeError('WANDB_API_KEY is missing or empty in Kaggle Secrets')
os.environ['WANDB_API_KEY'] = wandb_key.strip()
wandb.login(key=wandb_key.strip(), relogin=True)
del wandb_key
print('Repository, dependencies and W&B are ready.')

In [ ]:
# Hardware preflight: this notebook requires Kaggle GPU T4 x2.
import torch

gpu_count = torch.cuda.device_count()
print(f'Physical GPUs assigned by Kaggle: {gpu_count}')
for index in range(gpu_count):
    properties = torch.cuda.get_device_properties(index)
    print(f'GPU {index}: {properties.name}, {properties.total_memory / 2**30:.1f} GiB')
if gpu_count != 2:
    raise RuntimeError('Select GPU T4 x2 in Kaggle before starting the experiments.')
if GPUS != [0, 1]:
    raise ValueError('For Kaggle T4 x2, keep GPUS = [0, 1].')

concurrent = gpu_count * EXPERIMENTS_PER_GPU
print(f'Concurrent processes: {concurrent} ({EXPERIMENTS_PER_GPU} per GPU)')
if EXPERIMENTS_PER_GPU > 2:
    print('WARNING: four k=64 models may not fit or may be slower. Reduce EXPERIMENTS_PER_GPU after an OOM.')

In [ ]:
# Preview the exact launcher command.
command = [
    sys.executable, '-m', 'scripts.run_learning_curves',
    '--gpus', *[str(gpu) for gpu in GPUS],
    '--jobs-per-gpu', str(EXPERIMENTS_PER_GPU),
    '--max-experiments', str(TOTAL_EXPERIMENTS),
    '--k', str(K_UNITS),
    '--max-epochs', str(MAX_EPOCHS),
    '--validation-steps', str(LEARNING_CURVE_EVERY_STEPS),
    '--progress-steps', str(DASHBOARD_EVERY_STEPS),
    '--project', WANDB_PROJECT,
    '--output-dir', OUTPUT_ROOT,
]
print(' '.join(command))
subprocess.run([*command, '--dry-run'], cwd=repository, check=True)

## Start the campaign

The next cell blocks until the selected experiments finish. Stopping the cell also terminates all child training processes. Each slot bar is labelled with its physical GPU and experiment. Detailed logs, checkpoints and the machine-readable `run-status.json` are written under `OUTPUT_ROOT`.

In [ ]:
started = subprocess.run(command, cwd=repository)
print(f'Launcher exit code: {started.returncode}')
if started.returncode:
    print('At least one experiment failed. Inspect the summary and per-experiment logs below.')

In [ ]:
# Final summary. This cell can also be rerun while the campaign is active.
import json
import pandas as pd

run_directories = sorted(Path(OUTPUT_ROOT).glob('curves-*'))
if not run_directories:
    raise FileNotFoundError(f'No run directory found under {OUTPUT_ROOT}')
latest_run = run_directories[-1]
status_file = latest_run / 'run-status.json'
status = json.loads(status_file.read_text(encoding='utf-8'))
print(f'Latest run: {latest_run}')
print(f"queued={status['queued']} completed={status['completed']} failed={status['failed']}")
print('GPU occupancy:', status['gpus'])
display(pd.DataFrame(status['active']))
display(pd.DataFrame(status['results']))